# 14 — Gate Calibration & Benchmarking

Consolidates legacy experiments 13, 31, 32, 33, 34, and 35.

1. **Sequential rotations** — compose via session ops (Exp 13)
2. **DRAG calibration** — optimize DRAG coefficient (Exp 31)
3. **AllXY** — error pattern analysis (Exp 32)
4. **Standard RB** — average Clifford gate fidelity (Exp 33)
5. **Interleaved RB (unselective)** — per-gate fidelity (Exp 34)
6. **Interleaved RB (selective)** — selective pulse fidelity (Exp 35)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    AllXY,
    DRAGCalibration,
    RandomizedBenchmarking,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="14_gate_calibration_benchmarking",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

coherence_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current qubit frequency: {float(attr.qb_fq) / 1e9:.6f} GHz")
if coherence_checkpoint is not None:
    print(f"Prior stage 06 status: {coherence_checkpoint['status']}")

## 2. Benchmarking Defaults

In [ ]:
APPLY_DRAG_CALIBRATION = True

# --- DRAG Calibration (Exp 31) ---
DRAG_N_AVG = 10000

# --- AllXY (Exp 32) ---
ALLXY_N_AVG = 10000

# --- Standard RB (Exp 33) ---
RB_N_CLIFFORDS = [1, 2, 4, 8, 16, 32, 64, 128, 256]
RB_N_RANDOM_SEQUENCES = 50
RB_N_AVG = 1000

# --- Interleaved RB (Exp 34, 35) ---
IRB_N_CLIFFORDS = [1, 2, 4, 8, 16, 32, 64, 128]
IRB_N_RANDOM_SEQUENCES = 50
IRB_N_AVG = 1000

print("Gate calibration & benchmarking settings:")
print(f"  apply DRAG calibration: {APPLY_DRAG_CALIBRATION}")
print(f"  DRAG n_avg: {DRAG_N_AVG}")
print(f"  AllXY n_avg: {ALLXY_N_AVG}")
print(f"  Standard RB: n_cliffords={RB_N_CLIFFORDS}, n_seq={RB_N_RANDOM_SEQUENCES}, n_avg={RB_N_AVG}")
print(f"  Interleaved RB: n_cliffords={IRB_N_CLIFFORDS}, n_seq={IRB_N_RANDOM_SEQUENCES}, n_avg={IRB_N_AVG}")

## 3. Sequential Rotations — Exp 13

Basic single-qubit gate sequences for visual validation.

In [ ]:
# Compose sequences of x90, x180, y90 etc. and measure
# to verify gate primitives produce expected states.

seq_test_result = session.sequential_rotation_test(
    n_avg=DRAG_N_AVG,
)

print("Sequential rotation test complete.")

## 4. DRAG Calibration — Exp 31

Optimize the DRAG coefficient to minimize leakage during single-qubit gates.

In [ ]:
drag_cal = DRAGCalibration(session)
drag_result = drag_cal.run(n_avg=DRAG_N_AVG)
drag_analysis = drag_cal.analyze(drag_result, update_calibration=True)
drag_cal.plot(drag_analysis)

drag_fit_ok = fit_quality_gate(drag_analysis, r_squared_min=0.80)

if drag_fit_ok:
    drag_patch, drag_patch_preview, drag_apply_result = preview_or_apply_patch_ops(
        session,
        reason="DRAG coefficient calibration",
        proposed_patch_ops=drag_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_DRAG_CALIBRATION,
    )
    if drag_apply_result is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
        if attr is None:
            raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")
else:
    print("WARNING: DRAG calibration fit quality gate FAILED — skipping apply.")

drag_coeff = float(drag_analysis.metrics.get("drag_coeff", np.nan))
print(f"Optimized DRAG coefficient: {drag_coeff:.4f}")

## 5. AllXY — Exp 32

21-pair AllXY gate sequence to diagnose systematic errors in single-qubit gates.

In [ ]:
allxy = AllXY(session)
allxy_result = allxy.run(n_avg=ALLXY_N_AVG)
allxy_analysis = allxy.analyze(allxy_result, update_calibration=False)
allxy.plot(allxy_analysis)

allxy_deviation = float(allxy_analysis.metrics.get("max_deviation", np.nan))
print(f"AllXY max deviation: {allxy_deviation:.4f}")

## 6. Standard Randomized Benchmarking — Exp 33

Measure average Clifford gate fidelity.

In [ ]:
rb = RandomizedBenchmarking(session)
rb_result = rb.run(
    n_cliffords=RB_N_CLIFFORDS,
    n_random_sequences=RB_N_RANDOM_SEQUENCES,
    n_avg=RB_N_AVG,
)
rb_analysis = rb.analyze(rb_result, update_calibration=False)
rb.plot(rb_analysis)

rb_fidelity = float(rb_analysis.metrics.get("avg_clifford_fidelity", np.nan))
print(f"Average Clifford gate fidelity: {rb_fidelity:.6f}")
print(f"Error per Clifford: {1.0 - rb_fidelity:.2e}")

## 7. Interleaved RB — Unselective Pulses — Exp 34

Interleaved RB with standard (unselective) single-qubit gates.

In [ ]:
irb_unsel = RandomizedBenchmarking(session)
irb_unsel_result = irb_unsel.run(
    n_cliffords=IRB_N_CLIFFORDS,
    n_random_sequences=IRB_N_RANDOM_SEQUENCES,
    n_avg=IRB_N_AVG,
    interleaved=True,
    interleaved_gate="x180",
)
irb_unsel_analysis = irb_unsel.analyze(irb_unsel_result, update_calibration=False)
irb_unsel.plot(irb_unsel_analysis)

irb_unsel_fidelity = float(irb_unsel_analysis.metrics.get("interleaved_gate_fidelity", np.nan))
print(f"Interleaved x180 gate fidelity: {irb_unsel_fidelity:.6f}")

## 8. Interleaved RB — Selective Pulses — Exp 35

Interleaved RB with Fock-number-selective pulses.

In [ ]:
irb_sel = RandomizedBenchmarking(session)
irb_sel_result = irb_sel.run(
    n_cliffords=IRB_N_CLIFFORDS,
    n_random_sequences=IRB_N_RANDOM_SEQUENCES,
    n_avg=IRB_N_AVG,
    interleaved=True,
    interleaved_gate="sel_x180",
)
irb_sel_analysis = irb_sel.analyze(irb_sel_result, update_calibration=False)
irb_sel.plot(irb_sel_analysis)

irb_sel_fidelity = float(irb_sel_analysis.metrics.get("interleaved_gate_fidelity", np.nan))
print(f"Interleaved sel_x180 gate fidelity: {irb_sel_fidelity:.6f}")

## 9. Save Checkpoint

In [ ]:
drag_applied = "drag_apply_result" in dir() and drag_apply_result is not None

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="14_gate_calibration_benchmarking",
    status="calibrated" if drag_applied else "characterized",
    summary="DRAG calibration, AllXY, standard RB, and interleaved RB for single-qubit gate benchmarking.",
    consumed_inputs={
        "drag_n_avg": DRAG_N_AVG,
        "allxy_n_avg": ALLXY_N_AVG,
        "rb_n_cliffords": RB_N_CLIFFORDS,
        "rb_n_random_sequences": RB_N_RANDOM_SEQUENCES,
        "rb_n_avg": RB_N_AVG,
    },
    persisted_outputs={
        "drag_applied": drag_applied,
    },
    advisory_outputs={
        "drag_coeff": drag_coeff if "drag_coeff" in dir() else None,
        "allxy_max_deviation": allxy_deviation if "allxy_deviation" in dir() else None,
        "rb_avg_clifford_fidelity": rb_fidelity if "rb_fidelity" in dir() else None,
        "irb_unsel_x180_fidelity": irb_unsel_fidelity if "irb_unsel_fidelity" in dir() else None,
        "irb_sel_x180_fidelity": irb_sel_fidelity if "irb_sel_fidelity" in dir() else None,
    },
    next_stage="15_qubit_state_tomography",
    notes=[
        "RB and IRB are characterization-only — no calibration patches applied.",
        "DRAG coefficient is the only calibrated parameter.",
    ],
    metrics={
        "drag_coeff": drag_coeff if "drag_coeff" in dir() else None,
        "rb_avg_clifford_fidelity": rb_fidelity if "rb_fidelity" in dir() else None,
    },
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")